In [ ]:
import pandas as pd
import os
import re
import string
import spacy
from nltk.corpus import stopwords

from transformers import pipeline, logging



HUGGING FACE WORKFLOW:

1. Encoder-Only:
- Sentiment Analysis
- Named Entity Recognition
2. Decoder-Only
- VLANText Generation
3. Encoder-Decoder
- Zero-Shot Classification
- Text Summarization

# 1. Sentiment Analysis:

In [2]:
%%timeit
sentiment_analyzer = pipeline(task = "sentiment-analysis",
                              model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
                              device=-1,
                              truncation = True)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

1.38 s ± 58.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [3]:
#%%timeit
# create full path
file_path = os.path.join(os.getcwd(),'all-data.csv')
pd.set_option('display.max_colwidth',None)
df = pd.read_csv(file_path,encoding="ISO-8859-1")
df.columns = ['lable','Text']
#print(df.head())

In [4]:
# %%timeit
# load spaCy English model
nlp = spacy.load("en_core_web_sm")

# text cleaning function using spaCy
def clean_text_spacy(text):
    # lowercase
    text = text.lower()
    # remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    # remove digits
    text = re.sub(r"\d+", "", text)
    # remove extra whitespace
    text = " ".join(text.split())
    # remove stopwords with spaCy
    doc = nlp(text)
    text = " ".join([token.lemma_ for token in doc if not token.is_stop])
    return text

# apply cleaning
df["spacy_cln_txt"] = df["Text"].apply(clean_text_spacy)
df.columns

Index(['lable', 'Text', 'spacy_cln_txt'], dtype='str')

#### SPEEDING UP TRANSFORMERS CODE

- For new Macs with an Apple Silicon chip (i.e., M3 chip), choose device='mps' to use the GPU
- For PCs with a dedicated GPU (i.e., gaming PCs), choose device='cuda' to use your GPU
- For older Macs and most PCs, you’ll only be able to use your CPU for computations (default)

In [5]:
%%time

from transformers import pipeline

sentiment_analyzer = pipeline(
    task = "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english", # 1. smaller model
    device=-1, # running on CPU
    truncation=True,
    use_fast=True # 2. faster tokenization
)

import torch
torch.set_num_threads(1) # 3. specify multi-threading

with torch.no_grad(): # 4. disable gradients
    sentiment_scores = df['spacy_cln_txt'].apply(sentiment_analyzer)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

CPU times: total: 1h 44min 58s
Wall time: 32min 31s


# 2. NAMED ENTITY RECOGNITION (NER)

Named Entity Recognition (NER) is used to find and label important information (people, places, organizations, dates, etc.) in text
- The default LLM for NER is BERT (encoder-only)

Named Entity Recognition (NER)—also known as entity extraction or entity chunking—is a technique in Natural Language Processing (NLP) that automatically identifies and categorizes key information (entities) within unstructured text.

Think of it as an **AI highlighter**. If you give a computer a sentence like **"Elon Musk founded SpaceX in 2002,"** NER allows the machine to recognize "Elon Musk" as a Person, "SpaceX" as an Organization, and "2002" as a Date.

Common Entity Categories:

NER systems are typically trained to recognize several standard types of entities:
- Person (PER): Names of individuals (e.g., "Albert Einstein").
- Organization (ORG): Companies, agencies, or institutions (e.g., "Google", "NASA").
- Location (LOC/GPE): Geographical areas like cities, countries, or regions (e.g., "Paris", "India").
- Time/Date: Specific dates or timeframes (e.g., "5th May 2025").
- Quantities: Percentages, monetary values, or measurements (e.g., "$100", "50%").

Modern NER systems use three main approaches to "understand" text:
- Dictionary-Based: The system checks text against a pre-existing list (dictionary) of known entities. This is precise but struggles with new or misspelled names.
- Rule-Based: Uses human-written patterns (e.g., "If a word starts with a capital letter and follows 'Mr.', it is likely a Person").
- Machine Learning/Deep Learning: Models are trained on massive datasets to learn the context surrounding words. This allows them to handle ambiguity, such as distinguishing between Amazon the company and the Amazon rainforest.

NER is a foundational step for many advanced AI tasks:
- Search Engines: Helps search engines like Google and Bing provide more relevant results by understanding the entities in your query.
- Customer Support: Automatically categorizes support tickets by product name or location to route them to the right team.
- Content Recommendations: Platforms like Netflix use it to suggest shows based on the actors or genres mentioned in descriptions.
- Healthcare: Extracts patient information, symptoms, and medications from clinical notes for faster analysis.

In [6]:
text = 'On 12 May 2026 at 10:16 PM in Sadar, \
Uttar Pradesh, Ramu, a senior engineer at Antrhopol, \
was found murdered near the company’s office, with \
    investigators noting a knife wound measuring 12 cm, \
        and evidence suggesting a financial dispute \
            involving ₹125,486.'

from transformers import pipeline

ner_analyzer = pipeline(
    task = "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    device=-1,
    aggregation_strategy="SIMPLE"
)


ner_analyzer(text)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[{'entity_group': 'LOC',
  'score': np.float32(0.99842125),
  'word': 'Sadar',
  'start': 30,
  'end': 35},
 {'entity_group': 'LOC',
  'score': np.float32(0.99931407),
  'word': 'Uttar Pradesh',
  'start': 37,
  'end': 50},
 {'entity_group': 'PER',
  'score': np.float32(0.9953683),
  'word': 'Ramu',
  'start': 52,
  'end': 56},
 {'entity_group': 'ORG',
  'score': np.float32(0.993678),
  'word': 'Antrhopol',
  'start': 79,
  'end': 88}]

In [7]:
# find the named entities in each review
ner_analyzer = pipeline(task = "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    device=-1, # run on CPU
    aggregation_strategy='SIMPLE')

# apply to all reviews
df['Named_Entities'] = df['spacy_cln_txt'].apply(lambda text:
    [entity['word'] for entity in ner_analyzer(text)])

# create a unique list of named entities
named_entities = list(set(df.Named_Entities.explode().dropna().tolist()))

# exclude subwords from the list
named_entities_clean = [entity for entity in named_entities if '#' not in entity]
sorted(named_entities_clean)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

['a',
 'activision',
 'adac m',
 'af',
 'affecto',
 'affecto o',
 'affectogenimap',
 'affectogenimap o',
 'africa',
 'ah',
 'ahlstrom',
 'airvana',
 'aktia',
 'al',
 'alaj',
 'alandsbanken',
 'alexandria',
 'allone',
 'alpina',
 'altadis',
 'am',
 'americ',
 'american',
 'amex',
 'an',
 'antti orkola',
 'anttil',
 'arcelormittal',
 'arvo vuorenmaa',
 'as',
 'asahi kasei',
 'aspo',
 'aspo o',
 'aspo plc',
 'aspocomp',
 'astana',
 'atria',
 'atria eesti',
 'au',
 'aulas',
 'aura capital',
 'austra',
 'australia',
 'australian',
 'baltic',
 'basware',
 'bavaria industriekapital',
 'bavellon',
 'belarus',
 'belttongroup plc',
 'bergq',
 'bertrand sciard',
 'biohit',
 'blue',
 'blyk',
 'br',
 'brand',
 'brazilian',
 'brinkab',
 'burt',
 'c',
 'cameco',
 'cap',
 'capitex kalmar',
 'cargotec',
 'caverion',
 'celtnieciba',
 'celular',
 'cencor',
 'cencorp',
 'ceo',
 'ceo a',
 'ceo er',
 'ceo matti perkonoja',
 'ceo mika vehvilæinen',
 'ceo nordea',
 'ceo poyry plc',
 'ceo tel',
 'chambers',
 '

#### NAMED ENTITY RECOGNITION (NER)

Mathematical Understanding
TODO

ASSIGNMENT:
1. Read in the children’s books data set
2. Apply NER to the Description column
3. Create a list of all named entities
4. Only include the people (PER)
5. Extra credit: Exclude the authors as well

Zero-Shot Learning (ZSL):

Mathematical Understanding
TODO

**Definition**: A method where a model can classify or understand categories it has never seen during training, by leveraging auxiliary information such as text descriptions, semantic embeddings, or attributes.

**Key Idea**: Instead of training on labeled examples for every possible class, the model learns a **general representation** (often via language models or embeddings) that lets it reason about unseen classes.

Example:
- You train a model on animals like cats and dogs.
- Later, you ask it to classify zebras without zebra training data.
- The model uses semantic knowledge (e.g., “zebra = horse-like animal with stripes”) to make the prediction.

How Does Zero-Shot Learning Work?

- Semantic Space: Classes are described using attributes or natural language (e.g., “fruit that is yellow and curved” → banana).
- Embedding Alignment: Both input data (images, text) and class descriptions are mapped into the same embedding space.
- Inference: The model compares embeddings of unseen inputs with embeddings of class descriptions to assign labels.

TODO

ASSIGNMENT: ZERO-SHOT CLASSIFICATION

1. Apply zero-shot classification to the Description column
2. Find the number of books in each category and check a few to see if the results make sense

Mathematical Understanding:
TODO



#### TEXT SUMMARIZATION

Text summarization is used to make long bodies of text more concise
- The default LLM for text summarization is BART (encoder-decoder)

#### COSINE SIMILARITY:

#### what is meaning dot product between two vector?

The dot product (also called the scalar product or inner product) between two vectors is a way of measuring how much they “point in the same direction.”

- Measure Alignment: A positive result means they point in similar directions, while a negative result means they point in opposite directions.
- Identify Orthogonality: If the dot product is exactly zero, the vectors are perpendicular (at a $90^{\circ }$  angle).
- Calculate Projections: It helps determine how much of one vector exists in the direction of another.
- Find Angles: It provides a straightforward way to calculate the angle $\theta $  between two vectors in 2D or 3D space

- Mathematical Definition:
For two vectors in $\mathbb{R}^n$:
$$a \cdot b = \sum_{i=1}^{n} a_i b_i$$
* Multiply corresponding components.
* Add them up.
* Result is a **scalar** (single number).


Geometric Interpretation

The dot product can also be expressed in terms of magnitudes and the angle between vectors:

$$a \cdot b = \|a\| \|b\| \cos(\theta)$$

The geometric formulation for the cosine of the angle between two vectors can be written natively using standard vector notation:

$$\cos \theta = \frac{\vec{a} \cdot \vec{b}}{\|\vec{a}\| \|\vec{b}\|} ----- (i)$$


where

* **$a$ and $b$**: the $\vec{a}$ and $\vec{b}$ vectors.
* **$dp_{ab}$**: the $\vec{a} \cdot \vec{b}$ dot product between $a$ and $b$.
* **$dp_{aa}$**: the dot product of $a$ with itself.
* **$dp_{bb}$**: the dot product of $b$ with itself.
* **$l_a$ and $l_b$**: the $\|\vec{a}\|$ and $\|\vec{b}\|$ vector lengths.
* **$\hat{a}$ and $\hat{b}$**: unit vectors; i.e., $l_a = l_b = 1$.

#### Understand with Numerical Example

Given the vectors:
* $a = [1, 2, 3]$
* $b = [4, -5, 6]$

The dot products are calculated as follows:

* $a$ with $b$ is $dp_{ab} = 1 \cdot 4 + 2 \cdot (-5) + 3 \cdot 6 = 12$
* $a$ with itself is $dp_{aa} = 1 \cdot 1 + 2 \cdot 2 + 3 \cdot 3 = 14$
* $b$ with itself is $dp_{bb} = 4 \cdot 4 + (-5) \cdot (-5) + 6 \cdot 6 = 77$

#### Qus: What kind of information is stored in $dp_{ab}$, $dp_{aa}$, and $dp_{bb}$?

The sign of the dot product explicitly reveals the nature of the angle formed between vectors $a$ and $b$:

* **$dp_{ab} > 0$**: $a$ and $b$ form an angle less than $90^\circ$ (Acute angle).
* **$dp_{ab} = 0$**: $a$ and $b$ form an angle that is exactly $90^\circ$ (Orthogonal / Perpendicular).
* **$dp_{ab} < 0$**: $a$ and $b$ form an angle greater than $90^\circ$ (Obtuse angle).

Table 1. Possible types of angles


| Type      | acute     | right     | obtuse         | straight  | reflex         | perigon   |
|-----------|-----------|-----------|----------------|-----------|----------------|-----------|
| Angle     | θ < 90°   | θ = 90°   | 90° < θ < 180° | θ = 180°  | 180° < θ < 360°| θ = 360°  |

**Qus: What information is stored in $dp_{aa}$ and $dp_{bb}$?**

Taking square roots, it is clear that $dp_{aa}$ and $dp_{bb}$ hold *length* information. So:

* $l_a = (dp_{aa})^{1/2} = (14)^{1/2} = 3.74$ : i.e., the length of $a$.
* $l_b = (dp_{bb})^{1/2} = (77)^{1/2} = 8.77$ : i.e., the length of $b$.
* $l_a \cdot l_b = (dp_{aa})^{1/2} \cdot (dp_{bb})^{1/2} = 32.83$ : i.e., the length product ($lp_{ab}$) of $a$ and $b$.

#### Cosine Formula Breakdown

Using the previously calculated metrics, the cosine of the angle between the vectors is evaluated as follows:

$$\cos \theta = \frac{dp_{ab}}{lp_{ab}} = \frac{dp_{ab}}{\sqrt{dp_{aa}} \sqrt{dp_{bb}}} = \frac{12}{32.84} = 0.37    ------ (ii)$$

#### Comparing Vectors of Different Lengths

To compare vectors of different lengths, these can be recomputed as *unit vectors*. A unit vector is computed by dividing its elements by its length. In other words, we write the previous vectors as:

$$\hat{a} = [1/3.74, \ 2/3.74, \ 3/3.74]$$
$$\hat{b} = [4/8.77, \ -5/8.77, \ 6/8.77]$$

where the hat ( $\hat{}$ ) denotes a unit vector. Since the new lengths are equal to 1, the cosine similarity between $\hat{a}$ and $\hat{b}$ is their dot product; hence:

$$\cos \theta = dp_{\hat{a}\hat{b}} = 0.37 \tag{3}   ------ (iii)$$

Expressions (i), (ii), and (iii) return the same result, confirming that a cosine is a ***judgment of the orientation of the vectors, independent of their length***.

***From i, ii & iii we proved that orientation of the vectors, independent of their length.***



#### What does $\cos \theta = 0$ mean?

First we need to understand Raw Variables as Vectors

Orthogonality in $n$-Dimensional Space

* Imagine two variables $X = (x_1, x_2, \dots, x_n)$ and $Y = (y_1, y_2, \dots, y_n)$.
* Treat them as vectors in $n$-dimensional space.
* If their dot product is zero:

$$X \cdot Y = \sum_{i=1}^{n} x_i y_i = 0$$

then they are **orthogonal (perpendicular)** in that raw coordinate system.

Why "Raw" Matters

* Raw variables include their mean offset.
* Example: Suppose $X = (2, 2)$, $Y = (-1, 1)$.
    * Dot product: $2(-1) + 2(1) = -2 + 2 = 0$.
    * They are perpendicular in raw space.
* But if you **subtract the mean (center them)**, the relationship can change — correlation is defined on **centered vectors**.

#### Exampe of Mean-Center the Vectors:

- Let's take two 3D vectors:
$$X_1 = (2, 2, 2), \quad X_2 = (1, 0, -1)$$
- Dot Product (Raw)
- $$X_1 \cdot X_2 = (2)(1) + (2)(0) + (2)(-1) = 2 + 0 - 2 = 0$$
    * **Dot product = 0** $\rightarrow$ **orthogonal in raw space**.
    * So geometrically, they are perpendicular.
- Mean-Center the Vectors
    * Mean of $X_1$: $\bar{X}_1 = \frac{2 + 2 + 2}{3} = 2$
    * Mean of $X_2$: $\bar{X}_2 = \frac{1 + 0 + (-1)}{3} = 0$
- Now subtract the mean to obtain the centered vectors:
$$X_1' = (2 - 2, \ 2 - 2, \ 2 - 2) = (0, 0, 0)$$
$$X_2' = (1 - 0, \ 0 - 0, \ -1 - 0) = (1, 0, -1)$$

- Correlation on Centered Vectors
    * $X_1'$ is the **zero vector** (all values collapsed to mean).
    * This means correlation is **undefined** (division by zero variance).
    * So although raw vectors looked orthogonal, once centered, one vector has no variability $\rightarrow$ correlation cannot be computed.

#### Uncorrelated vs. Orthogonal Vectors final statement

* **Uncorrelated vectors:**
    - Two variables $X, Y$ are uncorrelated **if and only if** their **mean-centered vectors** are perpendicular.
    - $$r = 0 \iff (X - \bar{X}) \cdot (Y - \bar{Y}) = 0$$

* **Orthogonal raw vectors:**
    - If two raw vectors (without centering) are orthogonal, i.e.
    - $$X \cdot Y = 0$$
    - this **does not** imply they are uncorrelated, because correlation is defined on centered variables.

So we can conclude that two vector are uncorrelated if and only if their centered
variables are perpendicular. Menace if raw variables are orthogonal i.e. perpendicular,
we can not state they are uncorrelated.

#### Orthogonal vs. Uncorrelated
- **Orthogonal raw variables**: Dot product = 0, but means are not removed.
- **Uncorrelated variables**: Correlation coefficient 𝑟=0, which requires mean-centering first.

It turns out that unlike vector length normalization, subtracting the mean from raw variables
can change the angle between the vectors. The following can happen to the raw variables:

- If **they are perpendicular**, after mean-centering can become **not** perpendicular so they are orthogonal, but not uncorrelated.
- If **they are not perpendicular**, after mean-centering can become perpendicular so they are uncorrelated, but **not** orthogonal.
- If **they are perpendicular and after mean-centering remain perpendicular**, they are orthogonal and uncorrelated.
- If **they are not perpendicular and after mean-centering remain not perpendicular**, they are neither orthogonal nor uncorrelated—although their angle can change.

# ***That’s why orthogonality ≠ uncorrelated unless you center***.

## **The mathematical relationships connecting Pearson's Correlation Coefficient ($r$ ), Cosine Similarity, and the Slope ($b$ ) of a linear regression curve:**

#### 1. Pearson's $r$ as Cosine Similarity of Mean-Centered Variables

By definition, Cosine Similarity measures the cosine of the angle $\theta$ between two non-zero vectors $\mathbf{A}$ and $\mathbf{B}$ in an inner product space:

$$\text{Cosine Similarity} = \cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|} = \frac{\sum_{i=1}^{n} A_i B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \sqrt{\sum_{i=1}^{n} B_i^2}}$$

Now, let us examine two raw data variables, $X$ and $Y$. Pearson's correlation coefficient formula is:

$$r = \frac{\sum_{i=1}^{n} (X_i - \bar{X})(Y_i - \bar{Y})}{\sqrt{\sum_{i=1}^{n} (X_i - \bar{X})^2} \sqrt{\sum_{i=1}^{n} (Y_i - \bar{Y})^2}}$$

#### The Transformation (Mean-Centering)

If we transform our raw variables into mean-centered variables, we subtract the sample mean ($\bar{X}$ and $\bar{Y}$) from each data point:

$$\mathbf{x}_c = X_i - \bar{X} \quad \text{and} \quad \mathbf{y}_c = Y_i - \bar{Y}$$

Substituting these mean-centered vectors $\mathbf{x}_c$ and $\mathbf{y}_c$ directly into the Cosine Similarity formula yields:

$$\text{Cosine Similarity}(\mathbf{x}_c, \mathbf{y}_c) = \frac{\sum_{i=1}^{n} (\mathbf{x}_c)_i (\mathbf{y}_c)_i}{\sqrt{\sum_{i=1}^{n} (\mathbf{x}_c)_i^2} \sqrt{\sum_{i=1}^{n} (\mathbf{y}_c)_i^2}} = \frac{\sum_{i=1}^{n} (X_i - \bar{X})(Y_i - \bar{Y})}{\sqrt{\sum_{i=1}^{n} (X_i - \bar{X})^2} \sqrt{\sum_{i=1}^{n} (Y_i - \bar{Y})^2}}$$

This expression is identical to Pearson's $r$. Thus, Pearson's correlation coefficient is mathematically equivalent to the **cosine similarity of mean-centered data vectors**.

#### 2. Pearson's $r$ as Cosine Similarity of Z-Scores

A $Z$-score (standardized score) goes one step further than mean-centering by dividing the centered value by its sample standard deviation ($s_x$ or $s_y$):

$$Z_{x,i} = \frac{X_i - \bar{X}}{s_x} \quad \text{and} \quad Z_{y,i} = \frac{Y_i - \bar{Y}}{s_y}$$

Where $s_x = \sqrt{\frac{\sum (X_i - \bar{X})^2}{n-1}}$ (sample standard deviation).

##### Geometric Unit Vectors

When you convert an entire data vector into $Z$-scores, you scale its geometric length (Euclidean norm) to a fixed constant:

$$\|\mathbf{Z}_x\| = \sqrt{\sum_{i=1}^{n} \left(\frac{X_i - \bar{X}}{s_x}\right)^2} = \sqrt{\frac{\sum (X_i - \bar{X})^2}{\frac{\sum (X_i - \bar{X})^2}{n-1}}} = \sqrt{n-1}$$

If we calculate the Cosine Similarity between the two $Z$-score vectors $\mathbf{Z}_x$ and $\mathbf{Z}_y$:

$$\text{Cosine Similarity}(\mathbf{Z}_x, \mathbf{Z}_y) = \frac{\mathbf{Z}_x \cdot \mathbf{Z}_y}{\|\mathbf{Z}_x\| \|\mathbf{Z}_y\|} = \frac{\sum Z_{x,i} Z_{y,i}}{\sqrt{n-1}\sqrt{n-1}} = \frac{\sum Z_{x,i} Z_{y,i}}{n-1}$$

This exact formula, $\frac{\sum Z_{x,i} Z_{y,i}}{n-1}$, represents the statistical definition of Pearson's $r$ calculated from standard scores. Because cosine similarity eliminates vector scale entirely, it returns the same value whether calculated from mean-centered variables or fully standardized $Z$-scores.

#### 3. Computing $r$ from the Regression Slope

In an unstandardized simple linear regression curve ($Y = b_0 + b_1 X$), the formula for the slope ($b_1$) of the line of best fit is dictated by covariance and variance:

$$b_1 = \frac{\text{Cov}(X,Y)}{\text{Var}(X)} = \frac{\sum_{i=1}^{n}(X_i - \bar{X})(Y_i - \bar{Y})}{\sum_{i=1}^{n}(X_i - \bar{X})^2}$$

Comparing this directly to the Pearson's $r$ formula shows they share an identical numerator (covariance) but have different denominators:

$$r = b_1 \cdot \left(\frac{s_x}{s_y}\right)$$

* **$s_x$**: Standard deviation of $X$
* **$s_y$**: Standard deviation of $Y$

#### The Standardized Case (Z-Scores)

If the variables are first transformed into $Z$-scores, the standard deviations of both variables contract to exactly $1$ ($s_{Z_x} = 1$ and $s_{Z_y} = 1$). 

Plugging these values into the slope-to-correlation equation:

$$b_{\text{standardized}} = r \cdot \left(\frac{s_{Z_x}}{s_{Z_y}}\right) = r \cdot \left(\frac{1}{1}\right) = r$$

Consequently, when mapping relationships using standardized variables, the **slope of the regression curve is precisely equal to Pearson's Correlation Coefficient ($r$)**.

# Summary Matrix

| Data State    | Metric             | Geometric Interpretation                  | Statistical Equivalence |
|---------------|--------------------|-------------------------------------------|-------------------------|
| Mean-Centered | Cosine Similarity  | cos(θ) of vectors shifted to origin        | Pearson's r             |
| Z-Scores      | Cosine Similarity  | Dot product of unit-scaled vectors         | Pearson's r             |
| Z-Scores      | Regression Slope   | Unit rate of change of the best-fit line   | Pearson's r             |

#### Vector Example:

Suppose we have two text documents represented as term frequency vectors:

* $A = [1, 2, 3]$
* $B = [2, 3, 4]$

### Step 1: Dot product

$$A \cdot B = (1 \cdot 2) + (2 \cdot 3) + (3 \cdot 4) = 2 + 6 + 12 = 20$$

### Step 2: Magnitudes

$$\|A\| = \sqrt{1^2 + 2^2 + 3^2} = \sqrt{1 + 4 + 9} = \sqrt{14}$$

$$\|B\| = \sqrt{2^2 + 3^2 + 4^2} = \sqrt{4 + 9 + 16} = \sqrt{29}$$

### Step 3: Cosine similarity

$$\text{Cosine Similarity} = \frac{20}{\sqrt{14} \cdot \sqrt{29}} = \frac{20}{\sqrt{406}} \approx \frac{20}{20.149} \approx 0.994$$

#### Example with Mean-Centered Cosine Similarity (Pearson Correlation)

When we subtract the mean from our vectors before calculating the cosine similarity, we are performing **mean-centering**. The cosine similarity of mean-centered vectors is mathematically identical to **Pearson's correlation coefficient ($r$)**.

### Step 1: Compute Means
* Mean of $A$: $\bar{A} = \frac{1 + 2 + 3}{3} = 2$
* Mean of $B$: $\bar{B} = \frac{2 + 3 + 4}{3} = 3$

### Step 2: Subtract Means (Mean-Centering)
$$A_c = [1 - 2, \ 2 - 2, \ 3 - 2] = [-1, \ 0, \ 1]$$
$$B_c = [2 - 3, \ 3 - 3, \ 4 - 3] = [-1, \ 0, \ 1]$$

### Step 3: Dot Product of Centered Vectors
$$A_c \cdot B_c = (-1 \cdot -1) + (0 \cdot 0) + (1 \cdot 1) = 1 + 0 + 1 = 2$$

### Step 4: Magnitudes of Centered Vectors
$$\|A_c\| = \sqrt{(-1)^2 + 0^2 + 1^2} = \sqrt{2}$$
$$\|B_c\| = \sqrt{(-1)^2 + 0^2 + 1^2} = \sqrt{2}$$

### Step 5: Centered Cosine Similarity
$$\text{Centered Cosine Similarity} = \frac{A_c \cdot B_c}{\|A_c\| \|B_c\|} = \frac{2}{\sqrt{2} \cdot \sqrt{2}} = \frac{2}{2} = 1.0$$

> **Conclusion:** Because the relative changes across dimensions match perfectly after shifting to the center, the vectors have a perfect linear correlation ($r = 1.0$).

http://www.minerazzi.com/tools/cosine-similarity/cosine-similarity-calculator.php

#### Compute cosine similarity between the sentences by using DistilBERT embeddings:
- Sentence 1: "Ram playing football in ground"
- Sentence 2: ""

In [8]:
from transformers import DistilBertTokenizer, DistilBertModel
import torch
import numpy as np

# Load tokenizer and model once (outside function for efficiency)
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')

def cosine_similarity_distilbert(s1: str, s2: str):
    # Tokenize both sentences
    inputs1 = tokenizer(s1, return_tensors="pt")
    inputs2 = tokenizer(s2, return_tensors="pt")

    # Get embeddings
    with torch.no_grad():
        emb1 = model(**inputs1).last_hidden_state
        emb2 = model(**inputs2).last_hidden_state

    # Mean pooling
    vec1 = emb1.mean(dim=1).squeeze().numpy()
    vec2 = emb2.mean(dim=1).squeeze().numpy()
    
    # Print embeddings (first 10 dimensions for readability)
    print(f"Embedding for Sentence 1 ({s1}): {vec1[:10]}")
    print(f"Embedding for Sentence 2 ({s2}): {vec2[:10]}")

    # Cosine similarity
    cos_sim = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    #print("Cosine similarity:", cos_sim)
    return cos_sim

def cosine_similarity_no_mean_pool_distilbert(s1: str, s2: str):
    # Tokenize both sentences
    inputs1 = tokenizer(s1, return_tensors="pt")
    inputs2 = tokenizer(s2, return_tensors="pt")

    # Get embeddings
    with torch.no_grad():
        emb1 = model(**inputs1).last_hidden_state
        emb2 = model(**inputs2).last_hidden_state

    # Use CLS token embedding (first token)
    vec1 = emb1[:, 0, :].squeeze().numpy()
    vec2 = emb2[:, 0, :].squeeze().numpy()

    # Print embeddings (first 10 dimensions for readability)
    print(f"CLS Embedding for Sentence 1 ({s1}): {vec1[:10]}")
    print(f"CLS Embedding for Sentence 2 ({s2}): {vec2[:10]}")

    # Cosine similarity
    cos_sim = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    print("Cosine similarity:", cos_sim)

    return cos_sim

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [9]:
# Example usage
s1 = "Ram playing football in ground"
s2 = "Bird flying in sky"
cosine_similearity = cosine_similarity_distilbert(s1, s2)
print(f'your cosine similearity with Mean=> {cosine_similearity}')

cosine_similearity = cosine_similarity_no_mean_pool_distilbert(s1, s2)
print(f'your cosine similearity NO Mean=> {cosine_similearity}')

Embedding for Sentence 1 (Ram playing football in ground): [ 0.2521073   0.0216024  -0.32440382 -0.2740135   0.10649325 -0.3709229
  0.08672033  0.58601654 -0.18516754  0.06652392]
Embedding for Sentence 2 (Bird flying in sky): [ 0.16888928  0.1030749  -0.28000015 -0.0731909   0.28895304 -0.27413738
 -0.11623007  0.42357144 -0.20896955 -0.4120159 ]
your cosine similearity with Mean=> 0.6811526417732239
CLS Embedding for Sentence 1 (Ram playing football in ground): [-0.14585438 -0.03779666 -0.3845117  -0.3583348  -0.10947173 -0.4332577
  0.28161463  0.5173308  -0.28511143 -0.13844945]
CLS Embedding for Sentence 2 (Bird flying in sky): [-0.2115958  -0.01560094 -0.15723643 -0.10414591 -0.10441858 -0.14869337
  0.20204958  0.4883455  -0.38609943 -0.28427705]
Cosine similarity: 0.824303
your cosine similearity NO Mean=> 0.8243029713630676


In [10]:
# Example usage
s1 = "Sell car is not profitable"
s2 = "Eat carrot good to health"
cosine_similearity = cosine_similarity_distilbert(s1, s2)
print(f'your cosine similearity with Mean=> {cosine_similearity}')

cosine_similearity = cosine_similarity_no_mean_pool_distilbert(s1, s2)
print(f'your cosine similearity NO Mean=> {cosine_similearity}')

Embedding for Sentence 1 (Sell car is not profitable): [ 0.10362332 -0.13533026  0.09883011  0.27848548  0.4136576  -0.23186353
 -0.007145    0.45339176  0.15695111 -0.0341641 ]
Embedding for Sentence 2 (Eat carrot good to health): [ 0.10528266  0.04353394 -0.03163384  0.10447211  0.10599775 -0.16612656
  0.18938306  0.5481784   0.06971683 -0.2802419 ]
your cosine similearity with Mean=> 0.7181273102760315
CLS Embedding for Sentence 1 (Sell car is not profitable): [-0.20351906  0.01744328  0.01060115  0.00729521  0.06853931 -0.10211125
  0.04735465  0.5996116  -0.01267226 -0.2437369 ]
CLS Embedding for Sentence 2 (Eat carrot good to health): [-0.25952485 -0.02535985 -0.09759054 -0.08060504 -0.21997589 -0.05951842
  0.2543875   0.5502106  -0.13937004 -0.20069289]
Cosine similarity: 0.9160991
your cosine similearity NO Mean=> 0.9160990715026855


In [11]:
from sentence_transformers import SentenceTransformer, util

# Load a Sentence-BERT model fine-tuned for similarity
model = SentenceTransformer('all-MiniLM-L6-v2')

def cosine_similarity_sbert(s1: str, s2: str):
    # Get sentence embeddings
    emb1 = model.encode(s1, convert_to_tensor=True)
    emb2 = model.encode(s2, convert_to_tensor=True)

    # Cosine similarity
    cos_sim = util.cos_sim(emb1, emb2).item()
    print(f"Sentence 1: {s1}")
    print(f"Sentence 2: {s2}")
    print("Cosine similarity:", cos_sim)

    return cos_sim

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
s1 = "Ram playing football in ground"
s2 = "Bird flying in sky"
cosine_similearity = cosine_similarity_sbert(s1, s2)
print(f'your cosine similearity with Mean=> {cosine_similearity}')

Sentence 1: Ram playing football in ground
Sentence 2: Bird flying in sky
Cosine similarity: 0.20713487267494202
your cosine similearity with Mean=> 0.20713487267494202


In [13]:
s1 = "Ram Playing Villon"
s2 = "Seta Playing football"
cosine_similearity = cosine_similarity_sbert(s1, s2)
print(f'your cosine similearity with Mean=> {cosine_similearity}')

Sentence 1: Ram Playing Villon
Sentence 2: Seta Playing football
Cosine similarity: 0.2801934480667114
your cosine similearity with Mean=> 0.2801934480667114


In [14]:
s1 = "Sell car is not profitable"
s2 = "Eat carrot good to health"
cosine_similearity = cosine_similarity_sbert(s1, s2)
print(f'your cosine similearity with Mean=> {cosine_similearity}')

Sentence 1: Sell car is not profitable
Sentence 2: Eat carrot good to health
Cosine similarity: 0.04173694923520088
your cosine similearity with Mean=> 0.04173694923520088
